# Federated (GradAvg) — Dual-Contrastive Objective

$$\mathcal{L} = \mathcal{L}_{\mathrm{SFT}} + \lambda\left(\mathcal{L}_{\mathrm{ans}} + \mathcal{L}_{\mathrm{ctx}}\right)$$

Both odds-ratio terms in one loss: answer contrast ($y^{+}$ vs $y^{-}$ under
$c^{+}$) and context contrast ($c^{+}$ vs $c^{-}$ under $y^{+}$), sharing a
single weight. In code: `loss = nll + ORPO_LAMBDA * (or_loss_ctx + or_loss_pref)`
with `ORPO_LAMBDA = 0.2`.

---

*Notation:* `q` query · `c+` relevant snippet · `c-` off-topic snippet from the same user · `y+` personalised answer · `y-` general answer


In [1]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip -q install -U datasets huggingface_hub transformers accelerate peft bitsandbytes rouge-score

    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
else:
    DRIVE_ROOT = Path(".")

if IN_COLAB:
    PP_ROOT = DRIVE_ROOT / "privacy_perserving_pllm"
else:
    _here = Path(".").resolve()
    PP_ROOT = next(
        (
            p for p in [_here, *_here.parents]
            if (p / "v2_personamem_persona_subsets").exists()
            or all((p / d).exists() for d in ("centralised", "evaluation", "fed_grad_avg", "zero_shot"))
        ),
        _here,
    )
SUBSET_NAME = "all_subset"
SUBSETS_DIR = PP_ROOT / "v2_personamem_persona_subsets"

RESULTS_DIR = PP_ROOT / SUBSET_NAME / "v2_personamem_federated_combined_orpo_gradavg_snippet"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Subset:", SUBSET_NAME)
print("Results will be saved to:", RESULTS_DIR.resolve())

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 121.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 50.3 MB/s eta 0:00:00
Mounted at /content/drive
Subset: all_subset
Results will be saved to: /content/drive/MyDrive/privacy_perserving_pllm/all_subset/v2_personamem_federated_combined_orpo_gradavg_snippet


In [2]:
import ast
import json
import random
from typing import Any, Dict, List

from tqdm.auto import tqdm

import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer

DATASET_NAME = "bowen-upenn/PersonaMem-v2"
CONFIG = "benchmark"
TEXT_SPLITS = ("train_text", "val_text", "benchmark_text")

SEED = 42
TOKENIZER_NAME = "Qwen/Qwen3-0.6B"
random.seed(SEED)


def load_split(split):
    if split not in TEXT_SPLITS:
        raise ValueError(f"split must be one of {TEXT_SPLITS}, got {split!r}")
    return load_dataset(DATASET_NAME, CONFIG, split=split)


def parse_user_query(raw):
    if isinstance(raw, dict):
        return str(raw.get("content", raw)).strip()
    if isinstance(raw, str):
        try:
            d = ast.literal_eval(raw)
            if isinstance(d, dict):
                return str(d.get("content", raw)).strip()
        except (ValueError, SyntaxError):
            pass
        return raw.strip()
    return str(raw).strip()


def parse_incorrect_answers(raw):
    if isinstance(raw, list):
        return [str(x) for x in raw]
    if hasattr(raw, "tolist"):
        return [str(x) for x in raw.tolist()]
    if isinstance(raw, str):
        try:
            val = ast.literal_eval(raw)
            if isinstance(val, list):
                return [str(x) for x in val]
        except (ValueError, SyntaxError):
            pass
        return [raw]
    return []

def get_snippet(row):
    val = row.get("related_conversation_snippet")
    if val is None:
        return ""
    return str(val).strip()

In [3]:
def load_subset(name, subsets_dir=SUBSETS_DIR):
    """Load a persona subset (ids + train/val rows) saved by create_persona_subsets.ipynb."""
    subset_dir = Path(subsets_dir) / name
    if not subset_dir.exists():
        raise FileNotFoundError(f"Subset not found: {subset_dir}. Run create_persona_subsets.ipynb first.")
    with open(subset_dir / "personas.json", "r", encoding="utf-8") as f:
        persona_ids = [int(p) for p in json.load(f)]
    tr = pd.read_parquet(subset_dir / "train.parquet")
    va = pd.read_parquet(subset_dir / "val.parquet")
    return sorted(persona_ids), tr, va

MIN_VAL_ROWS = 4
CLIENT_PERSONAS, train_df, val_df = load_subset(SUBSET_NAME)
NUM_CLIENTS = len(CLIENT_PERSONAS)
val_counts = val_df.groupby("persona_id").size()
print(f"train rows: {len(train_df):,} | val rows: {len(val_df):,}")
print(f"Subset '{SUBSET_NAME}': {NUM_CLIENTS} personas")
print(f"Client personas ({len(CLIENT_PERSONAS)}): first 10 = {CLIENT_PERSONAS[:10]} ...")
print(f"Val rows per selected persona (min/mean/max): "
      f"{val_counts[CLIENT_PERSONAS].min()}/{val_counts[CLIENT_PERSONAS].mean():.1f}/{val_counts[CLIENT_PERSONAS].max()}")

train rows: 3,870 | val rows: 714
Subset 'all_subset': 150 personas
Client personas (150): first 10 = [6, 9, 14, 18, 33, 41, 51, 56, 57, 73] ...
Val rows per selected persona (min/mean/max): 4/4.8/8


In [4]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is not available. This notebook needs a GPU.\n"
        "In Google Colab: Runtime → Change runtime type → Hardware accelerator → GPU "
        "(e.g. T4), then Runtime → Restart session and re-run from the top."
    )
print("GPU:", torch.cuda.get_device_name(0))
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    get_peft_model_state_dict,
    prepare_model_for_kbit_training,
    set_peft_model_state_dict,
)
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen3-0.6B"
MAX_SEQ_LEN = 4096
MAX_SNIPPET_TOKENS = 2048
MAX_ANSWER_TOKENS = 512
MAX_NEW_TOKENS = 512

START_MODE = "hf"
CONTINUE_FROM_EPOCH = 3
CONTINUE_EPOCHS = 1
GLOBAL_EPOCHS = 3
GLOBAL_LR = 1e-3
GLOBAL_WEIGHT_DECAY = 0.0
MAX_GRAD_NORM = 1.0
LOCAL_BATCH_SIZE = 4

ORPO_LAMBDA = 0.2
EVAL_BATCH_SIZE = 32
EVAL_TRAIN = False

NEGATIVE_CTX_STRATEGY = "own_history"
NEG_SNIPPET_COL = "negative_conversation_snippet"

model_tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if model_tok.pad_token is None:
    model_tok.pad_token = model_tok.eos_token
model_tok.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

KEEP_COLS = ["user_query", "correct_answer", "incorrect_answers", "related_conversation_snippet"]
if NEG_SNIPPET_COL in train_df.columns:
    KEEP_COLS = KEEP_COLS + [NEG_SNIPPET_COL]
elif NEGATIVE_CTX_STRATEGY == "own_history":
    print(
        f"[warning] '{NEG_SNIPPET_COL}' not found in train.parquet - re-run "
        "create_persona_subsets.ipynb to generate own-history negatives; "
        "falling back to same-/cross-persona sampling."
    )

def build_system_prompt():
    return (
        "You are a personalised assistant. Use the conversation snippet to find "
        "information or connections relevant to the question, then provide the answer."
    )

def truncate_snippet(snippet, max_tokens):
    ids = model_tok(snippet, add_special_tokens=False)["input_ids"]
    if len(ids) <= max_tokens:
        return snippet
    return model_tok.decode(ids[-max_tokens:], skip_special_tokens=True)

def build_user_content(row, snippet=None):
    snippet_text = truncate_snippet(
        snippet if snippet is not None else get_snippet(row),
        MAX_SNIPPET_TOKENS,
    )
    return (
        "Relevant snippet from our earlier conversation:\n"
        f"{snippet_text}\n\n"
        f"{parse_user_query(row['user_query'])}"
    )

def build_prompt_text(system_prompt, row, snippet=None):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": build_user_content(row, snippet=snippet)},
    ]
    return model_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def valid_ctx_rows(rows):
    out = []
    for row in rows:
        if get_snippet(row).strip() and str(row.get("correct_answer", "")).strip():
            out.append(row)
    return out

def valid_training_rows(rows):
    out = []
    for row in rows:
        if not get_snippet(row).strip() or not str(row.get("correct_answer", "")).strip():
            continue
        if not parse_incorrect_answers(row.get("incorrect_answers", [])):
            continue
        out.append(row)
    return out

def build_negative_snippet_pools(train_frame):
    pools: Dict[int, List[str]] = {}
    for pid, grp in train_frame.groupby("persona_id"):
        snippets = [get_snippet(r) for r in grp.to_dict("records") if get_snippet(r).strip()]
        pools[int(pid)] = snippets
    return pools

def sample_negative_snippet(persona_id, pos_snippet, pools, all_snippets, rng):
    if NEGATIVE_CTX_STRATEGY == "cross_persona":
        candidates = []
        for pid, snippets in pools.items():
            if pid == persona_id:
                continue
            candidates.extend(snippets)
        candidates = [s for s in candidates if s != pos_snippet]
        if not candidates:
            candidates = [s for s in all_snippets if s != pos_snippet]
    else:
        candidates = [s for s in all_snippets if s != pos_snippet]
    if not candidates:
        raise ValueError("No negative snippet candidates available.")
    return rng.choice(candidates)

snippet_pools = build_negative_snippet_pools(train_df)
all_snippets = [s for snippets in snippet_pools.values() for s in snippets]
neg_rng = random.Random(SEED)
neg_source_counts: Dict[str, int] = {}

client_data: Dict[Any, Dict[str, Any]] = {}
pooled_pairs: List[Dict[str, Any]] = []
neg_mapping_records: List[Dict[str, Any]] = []

for pid in CLIENT_PERSONAS:
    p_train = train_df[train_df["persona_id"] == pid].reset_index(drop=True)
    system_prompt = build_system_prompt()
    rows = p_train[KEEP_COLS].to_dict("records")
    valid_rows = valid_training_rows(rows)
    ctx_pairs = []
    for row in valid_rows:
        pos_snippet = get_snippet(row)
        if NEGATIVE_CTX_STRATEGY == "own_history":
            neg_snippet = str(row.get(NEG_SNIPPET_COL) or "").strip()
            neg_source = "own_history"
            if not neg_snippet:
                own = [s for s in snippet_pools.get(int(pid), []) if s and s != pos_snippet]
                if own:
                    neg_snippet = neg_rng.choice(own)
                    neg_source = "own_persona_fallback"
                else:
                    neg_snippet = sample_negative_snippet(int(pid), pos_snippet, snippet_pools, all_snippets, neg_rng)
                    neg_source = "cross_persona_fallback"
        else:
            neg_snippet = sample_negative_snippet(int(pid), pos_snippet, snippet_pools, all_snippets, neg_rng)
            neg_source = NEGATIVE_CTX_STRATEGY
        neg_source_counts[neg_source] = neg_source_counts.get(neg_source, 0) + 1
        item = {
            "system_prompt": system_prompt,
            "row": row,
            "neg_snippet": neg_snippet,
            "neg_source": neg_source,
            "persona_id": int(pid),
        }
        ctx_pairs.append(item)
        pooled_pairs.append(item)
        neg_mapping_records.append({
            "persona_id": int(pid),
            "user_query": parse_user_query(row["user_query"]),
            "pos_snippet_preview": pos_snippet[:160],
            "neg_snippet_preview": neg_snippet[:160],
        })
    client_data[pid] = {
        "rows": rows,
        "valid_rows": valid_rows,
        "ctx_pairs": ctx_pairs,
        "n": len(rows),
        "n_ctx_pairs": len(ctx_pairs),
        "system_prompt": system_prompt,
    }
    print(f"  persona {pid}: {len(rows)} train rows | {len(ctx_pairs)} combined ORPO pairs")

print(f"Pooled combined ORPO pairs: {len(pooled_pairs)}")
print("Negative-context sources:", neg_source_counts)
with open(RESULTS_DIR / "negative_context_mapping_preview.json", "w", encoding="utf-8") as f:
    json.dump(neg_mapping_records[:20], f, indent=2)

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

  persona 6: 21 train rows | 21 combined ORPO pairs
  persona 9: 27 train rows | 27 combined ORPO pairs
  persona 14: 27 train rows | 27 combined ORPO pairs
  persona 18: 22 train rows | 22 combined ORPO pairs
  persona 33: 25 train rows | 25 combined ORPO pairs
  persona 41: 26 train rows | 26 combined ORPO pairs
  persona 51: 23 train rows | 23 combined ORPO pairs
  persona 56: 27 train rows | 27 combined ORPO pairs
  persona 57: 33 train rows | 33 combined ORPO pairs
  persona 73: 28 train rows | 28 combined ORPO pairs
  persona 80: 22 train rows | 22 combined ORPO pairs
  persona 83: 33 train rows | 33 combined ORPO pairs
  persona 87: 23 train rows | 23 combined ORPO pairs
  persona 91: 28 train rows | 28 combined ORPO pairs
  persona 93: 24 train rows | 24 combined ORPO pairs
  persona 94: 32 train rows | 32 combined ORPO pairs
  persona 95: 30 train rows | 30 combined ORPO pairs
  persona 99: 38 train rows | 38 combined ORPO pairs
  persona 105: 34 train rows | 34 combined ORPO 

In [5]:
import torch.nn.functional as F


def load_continual_adapter(from_epoch):
    """Continual learning: load base model + a previously saved LoRA adapter (trainable)."""
    adapter_dir = RESULTS_DIR / "global" / f"adapter_epoch_{from_epoch}"
    if not adapter_dir.exists():
        raise FileNotFoundError(f"Adapter to continue from not found: {adapter_dir}")
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto",
        trust_remote_code=True, attn_implementation="sdpa",
    )
    base.config.use_cache = False
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
    base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    base.enable_input_require_grads()
    m = PeftModel.from_pretrained(base, str(adapter_dir), is_trainable=True)
    m.print_trainable_parameters()
    return m


def load_base_with_adapter():
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto",
        trust_remote_code=True, attn_implementation="sdpa",
    )
    base.config.use_cache = False
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
    base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    base.enable_input_require_grads()
    m = get_peft_model(base, lora_config)
    return m


def clone_state(model):
    return {k: v.detach().cpu().clone() for k, v in get_peft_model_state_dict(model).items()}


def _build_input_labels(prompt_text, answer_text):
    eos = model_tok.eos_token or ""
    prompt_ids = model_tok(prompt_text, add_special_tokens=False)["input_ids"]
    answer_ids = model_tok(
        str(answer_text) + eos,
        add_special_tokens=False, truncation=True, max_length=MAX_ANSWER_TOKENS,
    )["input_ids"]
    input_ids = (prompt_ids + answer_ids)[:MAX_SEQ_LEN]
    prompt_len = min(len(prompt_ids), MAX_SEQ_LEN)
    labels = ([-100] * prompt_len + input_ids[prompt_len:])[:MAX_SEQ_LEN]
    attention_mask = [1] * len(input_ids)
    return input_ids, attention_mask, labels



def _forward_logits(model, input_ids, attention_mask):
    """Autocast on CUDA; plain forward on CPU (should not happen after CUDA check)."""
    device_type = next(model.parameters()).device.type
    if device_type == "cuda":
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            return model(input_ids=input_ids, attention_mask=attention_mask).logits
    return model(input_ids=input_ids, attention_mask=attention_mask).logits

def sequence_logprob(model, prompt_text, answer_text):
    input_ids, attention_mask, labels = _build_input_labels(prompt_text, answer_text)
    device = next(model.parameters()).device
    ids = torch.tensor([input_ids], device=device)
    attn = torch.tensor([attention_mask], device=device)
    lab = torch.tensor([labels], device=device)
    logits = _forward_logits(model, ids, attn)
    shift_logits = logits[:, :-1, :].float()
    shift_labels = lab[:, 1:]
    valid = shift_labels != -100
    safe = shift_labels.masked_fill(~valid, 0)
    log_probs = F.log_softmax(shift_logits, dim=-1)
    token_logps = log_probs.gather(dim=-1, index=safe.unsqueeze(-1)).squeeze(-1)
    total = (token_logps * valid).sum()
    count = valid.sum().clamp(min=1)
    return total, count


def _log1mexp(x):
    return torch.log(-torch.expm1(x.clamp(max=-1e-6)))


def _or_loss(mean_chosen, mean_rejected):
    log_odds = (mean_chosen - _log1mexp(mean_chosen)) - (mean_rejected - _log1mexp(mean_rejected))
    return -F.logsigmoid(log_odds), log_odds


def combined_orpo_loss_for_pair(model, system_prompt, row, neg_snippet):
    """NLL + combined orpo OR + standard preference OR under (q,c+)."""
    chosen = str(row["correct_answer"])
    rejected = parse_incorrect_answers(row["incorrect_answers"])[0]
    prompt_pos = build_prompt_text(system_prompt, row, snippet=get_snippet(row))
    prompt_neg = build_prompt_text(system_prompt, row, snippet=neg_snippet)

    sum_pos_c, n_pos_c = sequence_logprob(model, prompt_pos, chosen)
    sum_neg_c, n_neg_c = sequence_logprob(model, prompt_neg, chosen)
    sum_pos_r, n_pos_r = sequence_logprob(model, prompt_pos, rejected)
    mean_pos_c = sum_pos_c / n_pos_c
    mean_neg_c = sum_neg_c / n_neg_c
    mean_pos_r = sum_pos_r / n_pos_r

    nll = -mean_pos_c
    or_loss_ctx, log_odds_ctx = _or_loss(mean_pos_c, mean_neg_c)
    or_loss_pref, log_odds_pref = _or_loss(mean_pos_c, mean_pos_r)
    loss = nll + ORPO_LAMBDA * (or_loss_ctx + or_loss_pref)

    parts = {
        "loss": float(loss.detach().cpu()),
        "nll": float(nll.detach().cpu()),
        "or_loss_ctx": float(or_loss_ctx.detach().cpu()),
        "or_loss_pref": float(or_loss_pref.detach().cpu()),
        "or_loss": float((or_loss_ctx + or_loss_pref).detach().cpu()),
        "log_odds_ctx": float(log_odds_ctx.detach().cpu()),
        "log_odds_pref": float(log_odds_pref.detach().cpu()),
        "mean_pos_c": float(mean_pos_c.detach().cpu()),
        "mean_neg_c": float(mean_neg_c.detach().cpu()),
        "mean_pos_r": float(mean_pos_r.detach().cpu()),
    }
    return loss, parts


class ClientBatchSampler:
    """Yields fixed-size batches of a client's combined-ORPO pairs, reshuffling on wrap-around."""

    def __init__(self, rows, batch_size, seed=SEED):
        self.rows = rows
        self.batch_size = batch_size
        self.rng = random.Random(seed)
        self.order: List[int] = []
        self.pos = 0
        self._reshuffle()

    def _reshuffle(self):
        self.order = list(range(len(self.rows)))
        self.rng.shuffle(self.order)
        self.pos = 0

    def new_epoch(self):
        self._reshuffle()

    def next_batch(self):
        if not self.rows:
            return []
        batch = []
        while len(batch) < self.batch_size:
            if self.pos >= len(self.order):
                self._reshuffle()
            batch.append(self.rows[self.order[self.pos]])
            self.pos += 1
        return batch


def combined_orpo_batch_loss(model, batch_pairs):
    """Mean combined ORPO loss over a batch (keeps the graph for backward)."""
    losses = []
    parts_acc = {"nll": 0.0, "or_loss_ctx": 0.0, "or_loss_pref": 0.0, "or_loss": 0.0}
    for item in batch_pairs:
        loss, parts = combined_orpo_loss_for_pair(
            model, item["system_prompt"], item["row"], item["neg_snippet"]
        )
        losses.append(loss)
        for k in parts_acc:
            parts_acc[k] += parts[k]
    batch_loss = torch.stack(losses).mean()
    n = max(1, len(losses))
    return batch_loss, {k: v / n for k, v in parts_acc.items()}

In [6]:
import math

if START_MODE == "adapter":
    model = load_continual_adapter(CONTINUE_FROM_EPOCH)
    _start_epoch = CONTINUE_FROM_EPOCH + 1
    _end_epoch = CONTINUE_FROM_EPOCH + CONTINUE_EPOCHS
    print(f"Continual learning from adapter_epoch_{CONTINUE_FROM_EPOCH}: "
          f"training epochs {_start_epoch}..{_end_epoch}")
else:
    model = load_base_with_adapter()
    _start_epoch = 1
    _end_epoch = GLOBAL_EPOCHS
    print(f"Fresh HuggingFace base model: training epochs {_start_epoch}..{_end_epoch}")

model.train()
model.config.use_cache = False

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=GLOBAL_LR, weight_decay=GLOBAL_WEIGHT_DECAY)

active_personas = [pid for pid in CLIENT_PERSONAS if client_data[pid]["ctx_pairs"]]
num_active = len(active_personas)
samplers = {
    pid: ClientBatchSampler(client_data[pid]["ctx_pairs"], LOCAL_BATCH_SIZE)
    for pid in active_personas
}

steps_per_epoch = max(
    math.ceil(len(client_data[pid]["ctx_pairs"]) / LOCAL_BATCH_SIZE)
    for pid in active_personas
)
_num_train_epochs = _end_epoch - _start_epoch + 1
GLOBAL_STEPS = _num_train_epochs * steps_per_epoch
print(
    f"Active clients: {num_active}/{len(CLIENT_PERSONAS)} | steps/epoch: {steps_per_epoch} "
    f"| global steps: {GLOBAL_STEPS}"
)

GLOBAL_DIR = RESULTS_DIR / "global"
GLOBAL_DIR.mkdir(parents=True, exist_ok=True)
epoch_adapter_dirs: List[str] = []
training_log: Dict[str, Any] = {"method": "federated_combined_orpo_gradavg_global", "steps": []}

global_state = clone_state(model)
global_step = 0
for epoch in range(_start_epoch, _end_epoch + 1):
    for s in samplers.values():
        s.new_epoch()
    epoch_step_losses: List[float] = []

    for step in tqdm(range(1, steps_per_epoch + 1), desc=f"Epoch {epoch}/{_end_epoch}"):
        set_peft_model_state_dict(model, global_state)
        optimizer.zero_grad(set_to_none=True)
        client_step_losses: List[float] = []

        for pid in active_personas:
            batch = samplers[pid].next_batch()
            batch_loss, _ = combined_orpo_batch_loss(model, batch)
            (batch_loss / num_active).backward()
            client_step_losses.append(float(batch_loss.detach().cpu()))

        if MAX_GRAD_NORM:
            torch.nn.utils.clip_grad_norm_(params, MAX_GRAD_NORM)
        optimizer.step()

        global_state = clone_state(model)
        global_step += 1
        mean_step_loss = sum(client_step_losses) / len(client_step_losses)
        epoch_step_losses.append(mean_step_loss)
        training_log["steps"].append({
            "epoch": epoch, "step": step, "global_step": global_step,
            "mean_client_loss": mean_step_loss,
        })

    epoch_dir = GLOBAL_DIR / f"adapter_epoch_{epoch}"
    epoch_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(epoch_dir))
    model_tok.save_pretrained(str(epoch_dir))
    epoch_adapter_dirs.append(str(epoch_dir))
    print(
        f"epoch {epoch}/{_end_epoch} | updates this epoch={steps_per_epoch} "
        f"(global_step={global_step}) | mean_loss={sum(epoch_step_losses) / len(epoch_step_losses):.4f} "
        f"| saved {epoch_dir.name}"
    )

global_state = clone_state(model)

with open(GLOBAL_DIR / "training_losses.json", "w", encoding="utf-8") as f:
    json.dump(training_log, f, indent=2)

global_adapter_dir = GLOBAL_DIR / "adapter"
model.save_pretrained(str(global_adapter_dir))
model_tok.save_pretrained(str(global_adapter_dir))

with open(GLOBAL_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump({
        "method": "federated_combined_orpo_gradavg_global",
        "aggregation": "gradient_averaging_fedsgd",
        "single_global_optimizer": True,
        "subset": SUBSET_NAME,
        "client_personas": [int(p) for p in CLIENT_PERSONAS],
        "num_clients": len(CLIENT_PERSONAS),
        "num_active_clients": num_active,
        "min_val_rows": MIN_VAL_ROWS,
                "seed": SEED,
        "start_mode": START_MODE,
        "continue_from_epoch": CONTINUE_FROM_EPOCH if START_MODE == "adapter" else None,
        "continue_epochs": CONTINUE_EPOCHS if START_MODE == "adapter" else None,
        "trained_epoch_range": [int(_start_epoch), int(_end_epoch)],
        "global_epochs": GLOBAL_EPOCHS,
        "steps_per_epoch": steps_per_epoch,
        "global_steps": GLOBAL_STEPS,
        "global_lr": GLOBAL_LR,
        "global_weight_decay": GLOBAL_WEIGHT_DECAY,
        "max_grad_norm": MAX_GRAD_NORM,
        "local_batch_size": LOCAL_BATCH_SIZE,
        "orpo_lambda": ORPO_LAMBDA,
        "negative_ctx_strategy": NEGATIVE_CTX_STRATEGY,
        "loss": "nll(y+|q,c+) + lambda * -logsigmoid(log_odds(y+|q,c+) - log_odds(y+|q,c-))",
        "epoch_adapter_dirs": epoch_adapter_dirs,
        "final_adapter_dir": str(global_adapter_dir),
    }, f, indent=2)

del optimizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Saved global federated combined ORPO (gradient-averaging) adapter to:", global_adapter_dir.resolve())

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Fresh HuggingFace base model: training epochs 1..3
Active clients: 150/150 | steps/epoch: 10 | global steps: 30


...

epoch 1/3 | updates this epoch=10 (global_step=10) | mean_loss=3.0051 | saved adapter_epoch_1


Epoch 2/3:   0%|          | 0/10 [00:00<?, ?it/s]

epoch 2/3 | updates this epoch=10 (global_step=20) | mean_loss=2.4562 | saved adapter_epoch_2


Epoch 3/3:   0%|          | 0/10 [00:00<?, ?it/s]

epoch 3/3 | updates this epoch=10 (global_step=30) | mean_loss=2.0896 | saved adapter_epoch_3
Saved global federated combined ORPO (gradient-averaging) adapter to: /content/drive/MyDrive/privacy_perserving_pllm/all_subset/v2_personamem_federated_combined_orpo_gradavg_snippet/global/adapter


In [ ]:
def load_global_model(epoch=None):
    """Load the federated global combined ORPO LoRA adapter."""
    if epoch is not None:
        adapter_dir = RESULTS_DIR / "global" / f"adapter_epoch_{epoch}"
    else:
        adapter_dir = RESULTS_DIR / "global" / "adapter"
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto",
        trust_remote_code=True, attn_implementation="sdpa",
    )
    base.config.use_cache = True
    m = PeftModel.from_pretrained(base, str(adapter_dir))
    m.eval()
    tok = AutoTokenizer.from_pretrained(str(adapter_dir))
    return m, tok


print("Final global adapter:", RESULTS_DIR / "global" / "adapter")
print("Per-epoch checkpoints:", RESULTS_DIR / "global" / "adapter_epoch_<n>")
print("Next: evaluation/inference_global_adapter.ipynb then evaluation/rouge_eval_global_adapter.ipynb")